# Retrain EfficientDet for the Edge TPU with TensorFlow Lite Model Maker

这篇文章我们将介绍使用[TensorFlow Lite 模型制作工具](https://www.tensorflow.org/lite/guide/model_maker)基于[EfficientDet](https://ai.googleblog.com/2020/04/efficientdet-towards-scalable-and.html)训练一个EfficientDet-Lite对象识别模型。过程耗时30分钟左右。

我们使用DeepBot机器人采集的Mario图片和标注XML数据来作为训练数据，训练后的模型将能识别Mario。





## 安装依赖库

In [ ]:
!unzip dataset.zip

In [ ]:
!sudo apt-get update -y
!sudo apt-get install python3.9 python3.9-venv python3.9-distutils curl -y
# 创建虚拟环境（不会自带 pip）
!python3.9 -m venv /content/tflite_env
# 下载官方 get-pip 脚本
!curl -sS https://bootstrap.pypa.io/get-pip.py -o get-pip.py

# 使用虚拟环境中的 python 执行脚本安装 pip
!/content/tflite_env/bin/python get-pip.py
#验证 pip 是否生效
!/content/tflite_env/bin/pip --version

In [ ]:
!python3.9 -V
! /content/tflite_env/bin/pip install -q \
  tensorflow==2.10.0 \
  keras==2.10.0 \
  numpy==1.23.5 \
  protobuf==3.19.6 \
  tensorflow-hub==0.12.0 \
  tflite-support==0.4.2 \
  tensorflow-datasets==4.8.3 \
  sentencepiece==0.1.99 \
  sounddevice==0.4.5 \
  librosa==0.8.1 \
  flatbuffers==23.5.26 \
  matplotlib==3.5.3 \
  opencv-python==4.8.0.76

In [ ]:
! /content/tflite_env/bin/pip install tflite-model-maker==0.4.2

In [ ]:
! /content/tflite_env/bin/pip install matplotlib_inline IPython

In [ ]:
! /content/tflite_env/bin/pip install pycocotools

In [ ]:
! /content/tflite_env/bin/python -c "from pycocotools.coco import COCO; print('OK')"

In [ ]:
! /content/tflite_env/bin/python -c "from tflite_model_maker import image_classifier; print('TFLite Model Maker 已成功导入')"


## 选择基础模型



Model Maker模型制作工具支持EfficientDet-Lite系统对象识别模型生成，生成的模型可以适配Edge Tpu。EfficientDet-Lite源与[EfficientDet](https://ai.googleblog.com/2020/04/efficientdet-towards-scalable-and.html)，在小参数量模型中提供最先进的对象识别精度。下面是可以选择的预训练模型：

|| Model architecture | Size(MB)* | Latency(ms)** | Average Precision*** |
|-|--------------------|-----------|---------------|----------------------|
|| EfficientDet-Lite0 | 5.7       | 37.4            | 30.4%               |
|| EfficientDet-Lite1 | 7.6       | 56.3            | 34.3%               |
|| EfficientDet-Lite2 | 10.2      | 104.6           | 36.0%               |
|| EfficientDet-Lite3 | 14.4      | 107.6           | 39.4%               |
| <td colspan=4><br><i>* File size of the compiled Edge TPU models. <br/>** Latency measured on a desktop CPU with a Coral USB Accelerator. <br/>*** Average Precision is the mAP (mean Average Precision) on the COCO 2017 validation dataset.</i></td> |


Lite2和Lite3模型不适合Edge TPU运行，由于内存限制，上述两种模型推理时延会比较高。本次我们选择Lite0:

In [ ]:
spec = object_detector.EfficientDetLite0Spec()

In [ ]:
# step_train.py
with open('/content/step_train.py', 'w') as f:
    f.write("""
import tensorflow as tf
from tflite_model_maker import object_detector
from tflite_model_maker.config import ExportFormat

assert tf.__version__.startswith('2')

tf.get_logger().setLevel('ERROR')
from absl import logging
logging.set_verbosity(logging.ERROR)

spec = object_detector.EfficientDetLite0Spec()
label_map = {1: 'mario'}
train_images_dir = 'dataset/train/images'
train_annotations_dir = 'dataset/train/annotations'
val_images_dir = 'dataset/validation/images'
val_annotations_dir = 'dataset/validation/annotations'
test_images_dir = 'dataset/test/images'
test_annotations_dir = 'dataset/test/annotations'
train_data = object_detector.DataLoader.from_pascal_voc(
    train_images_dir, train_annotations_dir, label_map=label_map)
validation_data = object_detector.DataLoader.from_pascal_voc(
    val_images_dir, val_annotations_dir, label_map=label_map)
test_data = object_detector.DataLoader.from_pascal_voc(
    test_images_dir, test_annotations_dir, label_map=label_map)
print(f'train count: {len(train_data)}')
print(f'validation count: {len(validation_data)}')
test_data._dataset = {
    "info": {"description": "patched test dataset"},
    "images": [{"id": i} for i in range(len(test_data))],
    "categories": [{"id": i + 1, "name": name} for i, name in enumerate(test_data.label_map)],
    "annotations": []
}
print(f'test count: {len(test_data)}')
model = object_detector.create(train_data=train_data,
                               model_spec=spec,
                               validation_data=validation_data,
                               epochs=50,
                               batch_size=10,
                               train_whole_model=True)
model.summary()
#model.evaluate(test_data)

TFLITE_FILENAME = 'efficientdet-lite-mario.tflite'
#LABELS_FILENAME = 'salad-labels.txt'
#model.export(export_dir='.', tflite_filename=TFLITE_FILENAME, label_filename=LABELS_FILENAME,
             #export_format=[ExportFormat.TFLITE, ExportFormat.LABEL])
model.export(export_dir='.', tflite_filename=TFLITE_FILENAME,
             export_format=[ExportFormat.TFLITE, ExportFormat.LABEL])
#model.evaluate_tflite(TFLITE_FILENAME, test_data)
""")
! /content/tflite_env/bin/python /content/step_train.py

## 模型测试

安装依赖 [PyCoral API](https://coral.ai/docs/reference/py/):

In [ ]:
! python3 -m pip install --extra-index-url https://google-coral.github.io/py-repo/ pycoral

In [ ]:
! /content/tflite_env/bin/pip install --extra-index-url https://google-coral.github.io/py-repo/ pycoral

In [ ]:
# step_train.py
with open('/content/test_model.py', 'w') as f:
    f.write("""
from PIL import Image
from PIL import ImageDraw
from PIL import ImageFont

import tflite_runtime.interpreter as tflite
from pycoral.adapters import common
from pycoral.adapters import detect
from pycoral.utils.dataset import read_label_file
import numpy as np
import tensorflow as tf
import cv2

assert tf.__version__.startswith('2')

classes = ['mario']
# Define a list of colors for visualization
COLORS = np.random.randint(0, 255, size=(len(classes), 3), dtype=np.uint8)

def preprocess_image(image_path, input_size):
    img = tf.io.read_file(image_path)
    img = tf.io.decode_image(img, channels=3)
    img = tf.image.convert_image_dtype(img, tf.uint8)
    original_image = img
    resized_img = tf.image.resize(img, input_size)
    resized_img = resized_img[tf.newaxis, :]
    resized_img = tf.cast(resized_img, dtype=tf.uint8)
    return resized_img, original_image

def detect_objects(interpreter, image, threshold):

    signature_fn = interpreter.get_signature_runner()
    # Feed the input image to the model
    output = signature_fn(images=image)
    # Get all outputs from the model
    count = int(np.squeeze(output['output_0']))
    scores = np.squeeze(output['output_1'])
    classes = np.squeeze(output['output_2'])
    boxes = np.squeeze(output['output_3'])

    results = []
    for i in range(count):
        if scores[i] >= threshold:
            result = {
                'bounding_box': boxes[i],
                'class_id': classes[i],
                'score': scores[i]
            }
            results.append(result)
    return results

def run_odt_and_draw_results(image_path, interpreter, threshold=0.5):
    # Load the input shape required by the model
    _, input_height, input_width, _ = interpreter.get_input_details()[0]['shape']

    # Load the input image and preprocess it
    preprocessed_image, original_image = preprocess_image(
        image_path,
        (input_height, input_width)
    )

    # Run object detection on the input image
    results = detect_objects(interpreter, preprocessed_image, threshold=threshold)

    # Plot the detection results on the input image
    original_image_np = original_image.numpy().astype(np.uint8)
    print(results)
    for obj in results:
        # Convert the object bounding box from relative coordinates to absolute
        # coordinates based on the original image resolution
        ymin, xmin, ymax, xmax = obj['bounding_box']
        xmin = int(xmin * original_image_np.shape[1])
        xmax = int(xmax * original_image_np.shape[1])
        ymin = int(ymin * original_image_np.shape[0])
        ymax = int(ymax * original_image_np.shape[0])

        # Find the class index of the current object
        class_id = int(obj['class_id'])

        # Draw the bounding box and label on the image
        color = [int(c) for c in COLORS[class_id]]
        cv2.rectangle(original_image_np, (xmin, ymin), (xmax, ymax), color, 2)
        # Make adjustments to make the label visible for all objects
        y = ymin - 15 if ymin - 15 > 15 else ymin + 15
        label = "{}: {:.0f}%".format(classes[class_id], obj['score'] * 100)
        cv2.putText(original_image_np, label, (xmin, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # Return the final image
    original_uint8 = original_image_np.astype(np.uint8)
    return original_uint8



if __name__ == '__main__':
    #start to train

    #start to load the tflite to predict a image
    DETECTION_THRESHOLD = 0.3
    model_path = '/content/efficientdet-lite-mario.tflite'
    TEMP_FILE = '/content/dataset/train/images/snapshot_1753587088.jpg'

    # Load the TFLite model
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()

    # Run inference and draw detection result on the local copy of the original file
    detection_result_image = run_odt_and_draw_results(
        TEMP_FILE,
        interpreter,
        threshold=DETECTION_THRESHOLD
    )

    # save the detection result
    cv2.imwrite("detection_result.jpg", detection_result_image)

""")
! /content/tflite_env/bin/python /content/test_model.py

## 编译成Edge TPU




下载Edge TPU编译器:

In [ ]:
! curl https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo apt-key add -

! echo "deb https://packages.cloud.google.com/apt coral-edgetpu-stable main" | sudo tee /etc/apt/sources.list.d/coral-edgetpu.list

! sudo apt-get update

! sudo apt-get install edgetpu-compiler

In [ ]:
NUMBER_OF_TPUS =  1
TFLITE_FILENAME = 'efficientdet-lite-mario.tflite'
!edgetpu_compiler $TFLITE_FILENAME -d --num_segments=1

## 下载TPU文件

In [ ]:
from google.colab import files

files.download(TFLITE_FILENAME)
files.download(TFLITE_FILENAME.replace('.tflite', '_edgetpu.tflite'))